# 👁️ AI Smart Camera v0.2.0: FastAPI + React + Real-Time Computer Vision

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tahadeab-ai/ai-smart-camera/blob/main/notebooks/01_ai_smart_camera_colab.ipynb)
[![GitHub License](https://img.shields.io/badge/License-MIT-blue.svg)](https://github.com/tahadeab-ai/ai-smart-camera/blob/main/LICENSE)
[![FastAPI](https://img.shields.io/badge/Backend-FastAPI-009688.svg)](https://fastapi.tiangolo.com/)
[![React 18](https://img.shields.io/badge/Frontend-React%2018%20%2B%20TypeScript-61dafb.svg)](https://react.dev/)
[![YOLO](https://img.shields.io/badge/Model-YOLOv8%20%7C%20YOLO11-orange.svg)](https://docs.ultralytics.com/)
[![Supervision](https://img.shields.io/badge/Vision-Supervision-green.svg)](https://supervision.roboflow.com/)

Welcome to the official **Google Colab Interactive Execution Environment** for **AI Smart Camera v0.2.0**.

This notebook guides you through hardware inspection, repository setup, React production build, checksum-verified model loading, REST API tests, WebSocket camera streaming, automated pytest verification, reproducible benchmarking, and public HTTPS deployment via Cloudflare Tunnel.

---

## 1. Environment & Hardware Inspection
Inspect the Colab runtime environment, including CPU, RAM, and CUDA GPU acceleration (with automatic CPU fallback).

In [ ]:
import sys
import os
import platform
import psutil
import torch

print("=" * 60)
print("AI SMART CAMERA v0.2.0 - HARDWARE & ENVIRONMENT AUDIT")
print("=" * 60)
print(f"Python Version   : {platform.python_version()} ({platform.python_implementation()})")
print(f"Operating System : {platform.system()} {platform.release()}")
print(f"CPU Logical Cores: {psutil.cpu_count(logical=True)}")
ram_gb = psutil.virtual_memory().total / (1024 ** 3)
print(f"System Memory    : {ram_gb:.2f} GB RAM")
print(f"PyTorch Version  : {torch.__version__}")

cuda_available = torch.cuda.is_available()
print(f"CUDA Accelerated : {cuda_available}")
if cuda_available:
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
    print(f"GPU Model        : {gpu_name} ({gpu_mem:.2f} GB VRAM)")
    selected_device = "cuda"
else:
    print("GPU Model        : None detected (Using high-performance CPU inference)")
    selected_device = "cpu"

print(f"Active Device    : {selected_device.upper()}")
print("=" * 60)

## 2. Repository Setup & Dependency Installation
Clone the official repository and install pinned Python backend and test dependencies.

In [ ]:
import os
import sys

REPO_URL = "https://github.com/tahadeab-ai/ai-smart-camera.git"
PROJECT_DIR = "/content/ai-smart-camera"

if not os.path.exists(PROJECT_DIR):
    print(f"Cloning repository from {REPO_URL}...")
    !git clone {REPO_URL} {PROJECT_DIR}
else:
    print("Repository exists. Pulling latest code...")
    !cd {PROJECT_DIR} && git pull

%cd {PROJECT_DIR}
if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)

print("Installing backend and testing dependencies...")
!pip install -q -r requirements-dev.txt

import fastapi
import uvicorn
import ultralytics
import supervision as sv
import cv2

print("\nVerified Installed Core Packages:")
print(f"- FastAPI     : {fastapi.__version__}")
print(f"- Uvicorn     : {uvicorn.__version__}")
print(f"- Ultralytics : {ultralytics.__version__}")
print(f"- Supervision : {sv.__version__}")
print(f"- OpenCV      : {cv2.__version__}")

## 3. Node.js & React Frontend Production Build
Install Node.js dependencies and compile the React 18 + TypeScript SPA into `frontend/dist` for FastAPI single-origin serving.

In [ ]:
import subprocess
import os

print("Checking Node.js environment...")
!node -v && npm -v

print("\nInstalling frontend dependencies and building production SPA...")
!cd frontend && npm install && npm run build

dist_exists = os.path.exists("frontend/dist/index.html")
print(f"Frontend production build verified: {dist_exists}")
if dist_exists:
    !ls -lh frontend/dist

## 4. Checksum-Verified Model Loading & Manifest Audit
Verify SHA-256 integrity against `models/manifest.yaml` and initialize the active model.

In [ ]:
from app.core.model_loader import ModelLoader

loader = ModelLoader(manifest_path="models/manifest.yaml", cache_dir="models/weights")
MODEL_NAME = "yolov8n"

print(f"Loading and verifying model: {MODEL_NAME}...")
model, metadata = loader.load(model_name=MODEL_NAME, device=selected_device)

print("\n--- Model Metadata ---")
for key, val in metadata.items():
    if key != "classes":
        print(f"{key:20}: {val}")
print(f"Supported Categories : {metadata.get('classes_count', 80)} classes")

## 5. End-to-End Object Detection & Detailed Bounding Box API Inspection
Run inference on standard test image `bus.jpg` using `CameraService.process_image_detailed`.

In [ ]:
import urllib.request
from PIL import Image
from IPython.display import display
from app.services.camera_service import CameraService

TEST_IMAGE_URL = "https://ultralytics.com/images/bus.jpg"
urllib.request.urlretrieve(TEST_IMAGE_URL, "test_bus.jpg")
input_img = Image.open("test_bus.jpg")

service = CameraService(model_name=MODEL_NAME, device=selected_device)
_, annotated_rgb, boxes, stats, latency = service.process_image_detailed(input_img, confidence_threshold=0.3)

print(f"Inference Latency : {latency:.2f} ms")
print(f"Total Detections  : {stats['total_count']}")
print("Class Breakdown   :", stats["class_counts"])
print(f"Detected Boxes    : {len(boxes)} items (e.g. {boxes[0]['class_name']}: {boxes[0]['confidence'] * 100:.1f}%)")

display(Image.fromarray(annotated_rgb).resize((640, 480)))

## 6. Automated Test Suite & Code Quality Audit
Run the complete unit and integration test suite (including frame-skipping persistence and WebSocket protocol tests) along with Ruff linting.

In [ ]:
print("Running Ruff Lint Check...")
!ruff check .

print("\nRunning Automated pytest Suite...")
!pytest -v tests/ --cov=app --cov-report=term-missing

## 7. Reproducible CPU & GPU Benchmarking
Execute 5 warm-up and 30 measured runs with statistical latency calculation (Mean, P50, P95, P99, FPS).

In [ ]:
!python benchmarks/benchmark.py --model yolov8n --device {selected_device} --warmup 5 --runs 30

report_path = "benchmarks/benchmark_report.md"
if os.path.exists(report_path):
    with open(report_path, "r") as f:
        print(f.read())

## 8. Launch FastAPI Server with React Frontend & WebSockets
Start the application in the background on port 7860 and test local health endpoints.

In [ ]:
import subprocess
import time
import requests
import secrets

# Optional: Generate temporary access token for public Colab tunnel
TUNNEL_ACCESS_TOKEN = secrets.token_hex(16)
os.environ["APP_ACCESS_TOKEN"] = TUNNEL_ACCESS_TOKEN
print(f"Temporary Colab Access Token: {TUNNEL_ACCESS_TOKEN}")
print("(Save this token to paste into the web interface's Key settings if prompted)")

# Start FastAPI server in background
server_process = subprocess.Popen(
    [sys.executable, "-m", "app.main", "--host", "0.0.0.0", "--port", "7860"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
)

print("Waiting for FastAPI startup...")
time.sleep(5)

try:
    res = requests.get("http://localhost:7860/api/v1/health/live", timeout=5)
    print(f"Liveness check: {res.status_code} - {res.json()}")
    res_ready = requests.get("http://localhost:7860/api/v1/health/ready", timeout=5)
    print(f"Readiness check: {res_ready.status_code} - {res_ready.json()}")
except Exception as e:
    print(f"Startup check warning: {e}")

## 9. Public Cloudflare Quick Tunnel (Mobile Camera Testing)
Create a secure public HTTPS URL using Cloudflare Tunnel (`cloudflared`) to test mobile browser camera access and real-time streaming.

In [ ]:
!curl -s -L https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -o /tmp/cloudflared
!chmod +x /tmp/cloudflared

import subprocess
import time
import re

tunnel_log = "/tmp/cloudflared.log"
tunnel_proc = subprocess.Popen(
    ["/tmp/cloudflared", "tunnel", "--url", "http://localhost:7860"],
    stdout=open(tunnel_log, "w"),
    stderr=subprocess.STDOUT,
)

print("Establishing Cloudflare Quick Tunnel...")
tunnel_url = None
for _ in range(25):
    time.sleep(1)
    if os.path.exists(tunnel_log):
        content = open(tunnel_log).read()
        match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", content)
        if match:
            tunnel_url = match.group(0)
            break

if tunnel_url:
    print("=" * 65)
    print(f"🎉 PUBLIC APPLICATION URL: {tunnel_url}")
    print(f"🔑 ACCESS TOKEN           : {TUNNEL_ACCESS_TOKEN}")
    print("=" * 65)
    print("Open this URL on your mobile phone or browser to test real-time camera streaming!")
else:
    print("Could not extract tunnel URL. Inspect /tmp/cloudflared.log")

## 10. Teardown & GitHub Release Checklist
Stop background servers and review release verification status.

In [ ]:
# To cleanly stop the background tunnel and server:
# tunnel_proc.terminate()
# server_process.terminate()
print("AI Smart Camera v0.2.0 verification pipeline complete!")